##CyberShield Pro — Cyber Risk Assessment & Threat Intelligence Platform

## CELL 1 — INSTALL DEPENDENCIES

In [1]:
# ============================================================
#  CELL 1 — INSTALL DEPENDENCIES
#  Run this cell ONCE when you first set up the project.
# ============================================================

import sys
import subprocess
import importlib


def install_if_missing(package_name, import_name=None):
    """Try to import a package. If it fails, pip install it.

    Args:
        package_name: The pip install name  (e.g. 'fpdf2')
        import_name:  The import name       (e.g. 'fpdf')  — defaults to package_name
    """
    import_name = import_name or package_name
    try:
        importlib.import_module(import_name)
        print(f'  ✅  {package_name} already installed')
    except ImportError:
        print(f'  📦  Installing {package_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name, '-q'])
        print(f'  ✅  {package_name} installed')


print('Checking Python packages...\n')

install_if_missing('streamlit')                   # Web dashboard framework
install_if_missing('plotly')                      # Interactive charts & visualizations
install_if_missing('fastapi')                     # REST API framework
install_if_missing('uvicorn')                     # ASGI server for FastAPI
install_if_missing('slowapi')                     # Rate limiting for FastAPI endpoints
install_if_missing('fpdf2', 'fpdf')               # PDF generation (pip=fpdf2, import=fpdf)
install_if_missing('requests')                    # HTTP requests for VirusTotal API
install_if_missing('pandas')                      # Data manipulation & analysis
install_if_missing('python-dotenv', 'dotenv')     # Load .env file into os.environ
install_if_missing('pydantic')                    # Data validation for FastAPI models
install_if_missing('python-nmap', 'nmap')         # Python wrapper for nmap scanner

# ── Check if nmap binary is installed as a system tool ──────────────────────
# nmap is NOT a Python package — it's a system binary.
# On Windows: https://nmap.org/download.html
# On Linux:   sudo apt-get install nmap
# On Mac:     brew install nmap
print()
nmap_check = subprocess.run(['nmap', '--version'], capture_output=True, text=True)
if nmap_check.returncode == 0:
    print(f'  ✅  nmap: {nmap_check.stdout.splitlines()[0]}')
else:
    print('  ❌  nmap NOT found!')
    print('     → Linux:   sudo apt-get install nmap')
    print('     → Mac:     brew install nmap')
    print('     → Windows: https://nmap.org/download.html')

print()
print('All dependency checks complete! ✅')
print('Next → run Cell 2 to load your API keys from u.env')

Checking Python packages...

  ✅  streamlit already installed
  ✅  plotly already installed
  ✅  fastapi already installed
  ✅  uvicorn already installed
  ✅  slowapi already installed
  ✅  fpdf2 already installed
  ✅  requests already installed
  ✅  pandas already installed
  ✅  python-dotenv already installed
  ✅  pydantic already installed
  ✅  python-nmap already installed

  ✅  nmap: Nmap version 7.98 ( https://nmap.org )

All dependency checks complete! ✅
Next → run Cell 2 to load your API keys from u.env


In [ ]:
## Checking weather my u.env loaded are not ok

In [2]:
from dotenv import load_dotenv
import os

load_dotenv("u.env")

VT_KEY       = os.getenv('VT_API_KEY', '')
API_KEY      = os.getenv('CYBERSCAN_API_KEY', 'dev-key')
TARGETS_RAW  = os.getenv('SCAN_TARGETS', 'scanme.nmap.org') # here iam Targetting the Urls
GMAIL_SENDER = os.getenv('GMAIL_SENDER', '')
GMAIL_PASS   = os.getenv('GMAIL_PASSWORD', '') # go to Your google account  from Passwords you Have create a Password u have to save it it will Disappers after it create 
GMAIL_RCPT   = os.getenv('GMAIL_RECIPIENT', '')

TARGETS = [t.strip() for t in TARGETS_RAW.split(',') if t.strip()]

print('─' * 50)
print('CONFIGURATION STATUS')
print('─' * 50)
print(f'VT API Key     : {"✅ Loaded" if VT_KEY else "❌ Missing — add VT_API_KEY to .env"}')
print(f'API Key        : {"✅ Loaded" if API_KEY != "dev-key" else "⚠️  Using default dev-key"}')
print(f'Scan Targets   : {TARGETS}')
print(f'Email Sender   : {GMAIL_SENDER if GMAIL_SENDER else "❌ Missing — add GMAIL_SENDER to .env"}')
print(f'Email Recipient: {GMAIL_RCPT if GMAIL_RCPT else "❌ Missing — add GMAIL_RECIPIENT to .env"}')
print(f'Email Password : {"✅ Set" if GMAIL_PASS else "❌ Missing — add GMAIL_PASSWORD to .env"}')
print('─' * 50)

──────────────────────────────────────────────────
CONFIGURATION STATUS
──────────────────────────────────────────────────
VT API Key     : ✅ Loaded
API Key        : ⚠️  Using default dev-key
Scan Targets   : ['scanme.nmap.org', 'scanme2.nmap.org', 'nmap.org']
Email Sender   : kandulavamsi933@gmail.com
Email Recipient: mango79930@gmail.com
Email Password : ✅ Set
──────────────────────────────────────────────────


## 🔍 Step 2 — Scanner Module

In [3]:
%%writefile modules/scanner.py
import subprocess
import xml.etree.ElementTree as ET
import requests
import os

SCAN_DIR = "scan_results"
os.makedirs(SCAN_DIR, exist_ok=True)

VT_BASE = "https://www.virustotal.com/api/v3/ip_addresses"

def run_nmap_scan(target: str) -> str:
    xml_file = os.path.join(SCAN_DIR, f"{target}.xml") # here iam chekin by -Pn  is that Host alive are not,service version Sc is get extra info show banners,config
    subprocess.run(
        [
            "nmap",
            "-Pn",
            "-sV",
            "-sC",
            "--open",
            "-p-",          # scan ALL 65535 ports
            "--min-rate", "1000",  # faster scan
            "-oX", xml_file,
            target
        ],
        capture_output=True,
        timeout=300
    )
    return xml_file
#here Scanning the data that  where i got from the xml by using ox from nmap
def parse_nmap_xml(xml_file: str) -> list:
    if not os.path.exists(xml_file):
        return []
    try:
        root = ET.parse(xml_file).getroot()
    except ET.ParseError:
        return []
    rows = []
    for host in root.findall("host"):
        addr_el = host.find("address")
        if addr_el is None:
            continue
        ip = addr_el.get("addr", "unknown")
        for port in host.findall(".//port"):
            state_el = port.find("state")
            state    = state_el.get("state", "unknown") if state_el is not None else "unknown"
            if state not in ("open", "filtered"):
                continue
            svc = port.find("service")
            rows.append({
                "ip":       ip,
                "port":     port.get("portid", "0"),
                "protocol": port.get("protocol", "tcp"),
                "state":    state,
                "service":  svc.get("name",    "unknown") if svc is not None else "unknown",
                "product":  svc.get("product", "")        if svc is not None else "",
                "version":  svc.get("version", "")        if svc is not None else "",
            })
    return rows
#here scanning  virustotal activity 
def check_virustotal(ip: str, api_key: str) -> dict:
    _default = {
        "malicious_reports": 0, "suspicious_count":  0,
        "harmless_count":    0, "community_score":   0,
        "country": "Unknown",  "network": "Unknown", "categories": "",
    }
    if not api_key:
        return _default
    try:
        resp = requests.get(
            f"{VT_BASE}/{ip}",
            headers={"x-apikey": api_key},
            timeout=10
        )
        if resp.status_code != 200:
            return _default
        attrs = resp.json()["data"]["attributes"]
        stats = attrs.get("last_analysis_stats", {})
        votes = attrs.get("total_votes", {})
        cats  = attrs.get("categories", {})
        return {
            "malicious_reports": int(stats.get("malicious",  0)),
            "suspicious_count":  int(stats.get("suspicious", 0)),
            "harmless_count":    int(stats.get("harmless",   0)),
            "community_score":   int(votes.get("harmless",   0)) - int(votes.get("malicious", 0)),
            "country":           attrs.get("country",  "Unknown"),
            "network":           attrs.get("network",  "Unknown"),
            "categories":        ", ".join(set(cats.values())) if cats else "",
        }
    except Exception:
        return _default

def run_full_pipeline(targets: list, vt_api_key: str) -> list:
    all_rows = []
    for target in targets:
        xml_path = run_nmap_scan(target)
        all_rows.extend(parse_nmap_xml(xml_path))
    unique_ips = list({row["ip"] for row in all_rows})
    vt_cache   = {ip: check_virustotal(ip, vt_api_key) for ip in unique_ips}
    for row in all_rows:
        row.update(vt_cache.get(row["ip"], {}))
    return all_rows

Overwriting modules/scanner.py


## 📊 Step 3 — Analyser Module

In [4]:
%%writefile modules/analyser.py
import pandas as pd
#here iam checking  what are servicerisks 
SERVICE_RISK = {
    "telnet": 10, "rdp": 9, "smb": 9, "ftp": 8, "vnc": 8,
    "mongodb": 8, "redis": 8, "mysql": 7, "mssql": 7, "postgresql": 7,
    "smtp": 4, "dns": 3, "ssh": 3, "http": 2, "https": 1,
}  # here iam Setting Up the Serviceports to it 
#Dangerious ports
DANGEROUS_PORTS = {
    "21","23","135","137","139","445",
    "1433","3306","3389","5432","5900","6379","27017"
}  # and also iam Checking this Dangerous ports to here 

HIGH_RISK_COUNTRIES = {"CN","RU","KP","IR","NG","UA","VN","RO"}  # high risk countries also too

RECOMMENDATIONS = {
    "telnet":     "DISABLE IMMEDIATELY — replace with SSH. Telnet sends passwords in plaintext.",
    "ftp":        "Replace with SFTP or FTPS. Plain FTP credentials are visible on the network.",
    "rdp":        "Restrict to VPN only. Enable Network Level Authentication. Monitor login attempts.",
    "vnc":        "Ensure strong password is set. Restrict access to VPN or trusted IPs only.",
    "smb":        "Block port 445 at the perimeter. Verify MS17-010 (WannaCry) patch is applied.",
    "ssh":        "Disable password auth — use SSH keys only. Consider non-default port.",
    "http":       "Redirect all traffic to HTTPS. Check for outdated web application frameworks.",
    "https":      "Verify TLS version (1.2 minimum). Check certificate expiry and cipher suite.",
    "mysql":      "This port should NOT be internet-facing. Move behind VPN or firewall rule.",
    "postgresql": "Should not be internet-facing. Restrict to localhost or VPN.",
    "mssql":      "Restrict to internal network. Audit SQL Server authentication logs.",
    "redis":      "Redis has no authentication by default. Bind to localhost or require AUTH.",
    "mongodb":    "Enable authentication and restrict external access.",
    "smtp":       "Ensure relay is restricted. Check for open relay configuration.",
    "dns":        "If not a DNS server, close port 53. Disable recursion if public-facing.",
}
DEFAULT_REC = "Review this service. Confirm it is required and limit access to authorised sources."

# adding the expouser score here 
def _exposure_score(row):
    svc   = str(row.get("service", "")).lower()
    port  = str(row.get("port",    "0"))
    state = str(row.get("state",   "open")).lower()
    score = SERVICE_RISK.get(svc, 0)
    if score == 0 and port in DANGEROUS_PORTS:
        score = 6
    if score == 0:
        score = 1
    if state == "filtered":
        score = max(0.5, score * 0.5)
    return min(10.0, float(score))


def _threat_score(row):
    malicious  = int(row.get("malicious_reports", 0))  # here checking the threat Score what are the Malicious and susicious  and Commuinty Score 
    suspicious = int(row.get("suspicious_count",  0))
    community  = int(row.get("community_score",   0))
    raw = (malicious * 2.0) + (suspicious * 0.5)
    if community < 0:
        raw += min(2.0, abs(community) * 0.1)
    return min(10.0, raw)


def _context_score(row):  # here iam checking like The country that Have high score like vulnerability
    score   = 0.0
    country = str(row.get("country",    "")).upper()
    cats    = str(row.get("categories", "")).lower()
    comm    = int(row.get("community_score", 0))
    if country in HIGH_RISK_COUNTRIES: score += 3.0
    if "malware"  in cats: score += 4.0
    if "phishing" in cats: score += 3.0
    if "spam"     in cats: score += 2.0
    if "botnet"   in cats: score += 3.5
    if comm < -5:          score += 1.0
    return min(10.0, score)


def _severity(score):  # here iam Checking like what are things are critical high and Medium Score 
    if score >= 7.0: return "Critical"
    if score >= 5.0: return "High"
    if score >= 3.0: return "Medium"
    return "Low"


def enrich_dataframe(df, vt_data=None):
    df = df.copy()
    if vt_data:
        vt_df = pd.DataFrame(vt_data).T.rename_axis("ip").reset_index() # here iaam using dataframe for Setting a Data in a organized in nrows and columns
        df    = df.merge(vt_df, on="ip", how="left")
    for col, default in [
        ("malicious_reports", 0), ("suspicious_count", 0),
        ("harmless_count",    0), ("community_score",  0),
        ("country", "Unknown"),   ("network", "Unknown"), ("categories", "")
    ]:
        if col not in df.columns:
            df[col] = default
    df["exposure_score"] = df.apply(_exposure_score, axis=1).round(2)
    df["threat_score"]   = df.apply(_threat_score,   axis=1).round(2)
    df["context_score"]  = df.apply(_context_score,  axis=1).round(2)
    df["risk_score"]     = (
        df["exposure_score"] * 0.40 +
        df["threat_score"]   * 0.40 +
        df["context_score"]  * 0.20
    ).round(2)
    df["severity"]       = df["risk_score"].apply(_severity)
    df["recommendation"] = df["service"].apply(
        lambda s: RECOMMENDATIONS.get(str(s).lower(), DEFAULT_REC)
    )
    return df

# Host Summary in DATA FRAME
def build_host_summary(df):
    agg = df.groupby("ip").agg(
        open_ports       = ("port",            "count"),
        max_risk         = ("risk_score",       "max"),
        max_exposure     = ("exposure_score",   "max"),
        max_threat       = ("threat_score",     "max"),
        malicious_total  = ("malicious_reports","max"),
        country          = ("country",          "first"),
        network          = ("network",          "first"),
        categories       = ("categories",       "first"),
        services         = ("service",          lambda x: ", ".join(sorted(set(x)))),
    ).reset_index()
    agg["overall_severity"] = agg["max_risk"].apply(_severity)
    return agg.sort_values("max_risk", ascending=False)

#it was the generating the host summary from the scanner module
def generate_summary(df, host_sum=None):
    if host_sum is None:
        host_sum = build_host_summary(df)
    crit_hosts = int((host_sum["overall_severity"] == "Critical").sum())
    high_hosts = int((host_sum["overall_severity"] == "High").sum())
    vt_flagged = int(df.groupby("ip")["malicious_reports"].max().gt(0).sum())
    max_risk   = float(df["risk_score"].max()) if len(df) else 0.0
    if   crit_hosts > 0: posture, colour = "CRITICAL", "#7f1d1d"
    elif high_hosts > 0: posture, colour = "HIGH RISK", "#dc2626"
    elif max_risk  >= 3: posture, colour = "MODERATE", "#d97706"
    else:                posture, colour = "LOW RISK",  "#16a34a"
    findings = [] # finding the all the things in data Frame
    plaintext = df[df["service"].isin(["telnet","ftp"])]
    if len(plaintext):
        findings.append(f"Plaintext protocol(s) detected: {', '.join(plaintext['service'].unique())}")
    flagged_ips = df[df["malicious_reports"] > 0]["ip"].nunique()
    if flagged_ips:
        findings.append(f"{flagged_ips} IP(s) flagged as malicious by VirusTotal engines")
    db_ports = df[df["service"].isin(["mysql","mssql","postgresql","mongodb","redis"])]
    if len(db_ports):
        findings.append(f"Database port(s) exposed: {db_ports['service'].unique().tolist()}")
    risky_countries = df[df["country"].isin({"CN","RU","KP","IR","NG","UA","VN","RO"})]["ip"].nunique()
    if risky_countries:
        findings.append(f"{risky_countries} IP(s) registered in high-risk countries")
    suspicious = df[df["suspicious_count"] > 0]["ip"].nunique()
    if suspicious:
        findings.append(f"{suspicious} IP(s) have suspicious (unconfirmed) VT flags")
    rdp_smb = df[df["service"].isin(["rdp","smb","ms-wbt-server"])]
    if len(rdp_smb):
        findings.append(f"Remote access port(s) open: {rdp_smb['service'].unique().tolist()}")
    if not findings:
        findings.append("No critical findings detected in this scan")
    return {
        "total_hosts": int(df["ip"].nunique()),
        "total_ports": len(df),
        "crit_hosts":  crit_hosts,
        "high_hosts":  high_hosts,
        "vt_flagged":  vt_flagged,
        "max_risk":    max_risk,
        "posture":     posture,
        "colour":      colour,
        "findings":    findings,
    }

Overwriting modules/analyser.py


## 💾 Step 4 — Database Module

In [5]:
%%writefile modules/database.py
import sqlite3
import pandas as pd
import json
from datetime import datetime
#it is the file where i wwas saving the file sto it 
DB_FILE = "cyberscan.db"


def init_db():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("""CREATE TABLE IF NOT EXISTS scans (
        id             INTEGER PRIMARY KEY AUTOINCREMENT,
        scan_time      TEXT    NOT NULL,
        targets        TEXT,
        total_hosts    INTEGER,
        total_ports    INTEGER,
        critical_count INTEGER,
        high_count     INTEGER,
        max_risk_score REAL,
        avg_risk_score REAL,
        results_json   TEXT
    )""")
    conn.commit()
    conn.close()


def save_scan(df, targets):
    conn = sqlite3.connect(DB_FILE)
    cur  = conn.execute(
        """INSERT INTO scans
           (scan_time, targets, total_hosts, total_ports,
            critical_count, high_count, max_risk_score, avg_risk_score, results_json)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
        (
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            ", ".join(targets),
            int(df["ip"].nunique()),
            len(df),
            int((df["severity"] == "Critical").sum()),
            int((df["severity"] == "High").sum()),
            float(df["risk_score"].max()),
            float(df["risk_score"].mean()),
            df.to_json(orient="records"),
        )
    )
    conn.commit()
    scan_id = cur.lastrowid
    conn.close()
    return scan_id


def load_history():
    conn = sqlite3.connect(DB_FILE)
    df   = pd.read_sql_query(
        """SELECT id, scan_time, targets, total_hosts, total_ports,
                  critical_count, high_count, max_risk_score, avg_risk_score
           FROM scans ORDER BY id DESC""", conn
    )
    conn.close()
    return df


def load_scan_by_id(scan_id):
    conn = sqlite3.connect(DB_FILE)
    row  = conn.execute(
        "SELECT results_json FROM scans WHERE id = ?", (scan_id,)
    ).fetchone()
    conn.close()

    if not row or not row[0]:
        return pd.DataFrame()

    try:
        # use json.loads first then DataFrame
        data = json.loads(row[0])
        if not data:
            return pd.DataFrame()
        df = pd.DataFrame(data)
        return df
    except Exception as e:
        print(f'load_scan_by_id error: {e}')
        return pd.DataFrame()


init_db()

Overwriting modules/database.py


## 📧 Step 5 — Emailer Module

In [6]:
%%writefile modules/emailer.py
# modules/emailer.py
import smtplib
import os
from email.mime.text        import MIMEText
from email.mime.multipart   import MIMEMultipart
from email.mime.application import MIMEApplication
from datetime import datetime
import pandas as pd

REPORTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'reports')
REPORTS_DIR = os.path.normpath(REPORTS_DIR)
os.makedirs(REPORTS_DIR, exist_ok=True)

SEV_COLOURS = {
    'Critical': '#7f1d1d',
    'High':     '#dc2626',
    'Medium':   '#ea580c',
    'Low':      '#16a34a',
}


def _safe_str(val, max_len=None):
    """Convert any value to a plain ASCII string safe for fpdf Helvetica font."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        result = ''
    else:
        result = str(val)

    replacements = {
        '\u2014': '-',
        '\u2013': '-',
        '\u2012': '-',
        '\u2010': '-',
        '\u2018': "'",
        '\u2019': "'",
        '\u201c': '"',
        '\u201d': '"',
        '\u2026': '...',
        '\u2022': '*',
        '\u00b7': '.',
        '\u00a0': ' ',
    }
    for char, replacement in replacements.items():
        result = result.replace(char, replacement)

    result = result.encode('ascii', errors='ignore').decode('ascii')

    if max_len:
        result = result[:max_len]
    return result


def generate_pdf_report(df: pd.DataFrame, scan_time: str, output_path: str) -> str:
    from fpdf import FPDF

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # ── Header ────────────────────────────────────────────────────────────────
    pdf.set_fill_color(15, 17, 23)
    pdf.rect(0, 0, 210, 35, 'F')
    pdf.set_font('Helvetica', 'B', 20)
    pdf.set_text_color(255, 255, 255)
    pdf.set_y(10)
    pdf.cell(0, 10, 'CyberScan Pro - Scan Report', align='C', ln=True)
    pdf.set_font('Helvetica', '', 10)
    pdf.cell(0, 6, 'Generated: ' + _safe_str(scan_time), align='C', ln=True)
    pdf.ln(12)

    # ── Executive Summary ─────────────────────────────────────────────────────
    pdf.set_text_color(30, 30, 30)
    pdf.set_font('Helvetica', 'B', 12)
    pdf.cell(0, 8, 'Executive Summary', ln=True)
    pdf.set_font('Helvetica', '', 10)

    try:
        max_risk = '{:.2f} / 10.00'.format(df['risk_score'].max())
    except Exception:
        max_risk = 'N/A'

    try:
        vt_flagged = str(int((df['malicious_reports'] > 0).sum()))
    except Exception:
        vt_flagged = '0'

    kpis = [
        ('Hosts Scanned',     str(df['ip'].nunique())),
        ('Open Ports',        str(len(df))),
        ('Critical Findings', str(int((df['severity'] == 'Critical').sum()))),
        ('High Findings',     str(int((df['severity'] == 'High').sum()))),
        ('Max Risk Score',    max_risk),
        ('VT Flagged IPs',    vt_flagged),
    ]
    for label, val in kpis:
        pdf.cell(80, 7, _safe_str(label), border=1)
        pdf.cell(40, 7, _safe_str(val),   border=1, ln=True)
    pdf.ln(6)

    # ── Detailed Findings ─────────────────────────────────────────────────────
    pdf.set_font('Helvetica', 'B', 11)
    pdf.cell(0, 8, 'Detailed Findings (Critical + High)', ln=True)
    pdf.set_font('Helvetica', 'B', 8)

    headers = ['IP', 'Port', 'Service', 'Severity', 'Risk', 'Country', 'Recommendation']
    widths  = [30,    15,     20,        18,          12,     16,        79]

    for h, w in zip(headers, widths):
        pdf.set_fill_color(30, 58, 138)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(w, 7, h, border=1, fill=True)
    pdf.ln()

    pdf.set_font('Helvetica', '', 7)

    try:
        alert_df = df[df['severity'].isin(['Critical', 'High'])].sort_values(
            'risk_score', ascending=False
        )
    except Exception:
        alert_df = df

    for idx, (_, row) in enumerate(alert_df.iterrows()):
        bg = (249, 250, 251) if idx % 2 == 0 else (255, 255, 255)
        pdf.set_fill_color(*bg)
        pdf.set_text_color(30, 30, 30)

        try:
            risk_val = '{:.2f}'.format(float(row.get('risk_score', 0)))
        except Exception:
            risk_val = '0.00'

        vals = [
            _safe_str(row.get('ip',             ''), 30),
            _safe_str(row.get('port',           ''), 10),
            _safe_str(row.get('service',        ''), 15),
            _safe_str(row.get('severity',       ''), 12),
            risk_val,
            _safe_str(row.get('country',        ''), 12),
            _safe_str(row.get('recommendation', ''), 55),
        ]
        for val, w in zip(vals, widths):
            pdf.cell(w, 6, val, border=1, fill=True)
        pdf.ln()

    # ── Footer ────────────────────────────────────────────────────────────────
    pdf.ln(4)
    pdf.set_font('Helvetica', 'I', 8)
    pdf.set_text_color(100, 100, 100)
    pdf.cell(
        0, 6,
        'CyberScan Pro  |  Confidential  |  For authorised recipients only',
        align='C', ln=True
    )

    pdf.output(output_path)
    return output_path


def _build_html_body(df: pd.DataFrame, scan_time: str) -> str:
    alert_df = df.sort_values('risk_score', ascending=False)
    count    = len(alert_df)
    crit_cnt = int((df['severity'] == 'Critical').sum())
    high_cnt = int((df['severity'] == 'High').sum())
    med_cnt  = int((df['severity'] == 'Medium').sum())
    low_cnt  = int((df['severity'] == 'Low').sum())

    rows_html = ''
    for idx, (_, row) in enumerate(alert_df.iterrows()):
        bg  = '#f9fafb' if idx % 2 == 0 else '#ffffff'
        sev = str(row.get('severity', ''))
        col = SEV_COLOURS.get(sev, '#374151')
        rec = str(row.get('recommendation', ''))[:80]
        rows_html += (
            f'<tr style="background:{bg};">'
            f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;font-family:monospace;color:#1e40af;">{row.get("ip","")}</td>'
            f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">{row.get("port","")}</td>'
            f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">{row.get("service","")}</td>'
            f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;color:{col};font-weight:bold;">{sev}</td>'
            f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;text-align:center;font-weight:bold;">{float(row.get("risk_score",0)):.1f}</td>'
            f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;">{row.get("country","")}</td>'
            f'<td style="padding:8px 12px;border-bottom:1px solid #e5e7eb;font-size:12px;color:#374151;">{rec}</td>'
            f'</tr>'
        )

    return f'''
    <html><body style="font-family:Arial,sans-serif;background:#f3f4f6;margin:0;padding:20px;">
    <div style="max-width:900px;margin:0 auto;background:white;border-radius:8px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,.12);">
      <div style="background:#0f172a;padding:24px 32px;">
        <h1 style="color:white;margin:0;font-size:22px;">CyberScan Pro - Security Alert</h1>
        <p style="color:#94a3b8;margin:6px 0 0;">Scan completed: {scan_time}</p>
      </div>
      <div style="padding:24px 32px;">
        <div style="display:flex;gap:16px;margin-bottom:24px;">
          <div style="background:#fef2f2;border:1px solid #fecaca;border-radius:6px;padding:14px 20px;">
            <div style="font-size:28px;font-weight:bold;color:#7f1d1d;">{crit_cnt}</div>
            <div style="font-size:13px;color:#991b1b;">Critical</div>
          </div>
          <div style="background:#fff7ed;border:1px solid #fed7aa;border-radius:6px;padding:14px 20px;">
            <div style="font-size:28px;font-weight:bold;color:#dc2626;">{high_cnt}</div>
            <div style="font-size:13px;color:#c2410c;">High</div>
          </div>
          <div style="background:#fffbeb;border:1px solid #fde68a;border-radius:6px;padding:14px 20px;">
            <div style="font-size:28px;font-weight:bold;color:#d97706;">{med_cnt}</div>
            <div style="font-size:13px;color:#b45309;">Medium</div>
          </div>
          <div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:6px;padding:14px 20px;">
            <div style="font-size:28px;font-weight:bold;color:#166534;">{low_cnt}</div>
            <div style="font-size:13px;color:#15803d;">Low</div>
          </div>
          <div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:6px;padding:14px 20px;">
            <div style="font-size:28px;font-weight:bold;color:#1d4ed8;">{count}</div>
            <div style="font-size:13px;color:#1e40af;">Total</div>
          </div>
        </div>
        <table style="width:100%;border-collapse:collapse;font-size:13px;">
          <thead>
            <tr style="background:#1e3a5f;color:white;">
              <th style="padding:10px 12px;text-align:left;">IP Address</th>
              <th style="padding:10px 12px;">Port</th>
              <th style="padding:10px 12px;">Service</th>
              <th style="padding:10px 12px;">Severity</th>
              <th style="padding:10px 12px;">Score</th>
              <th style="padding:10px 12px;">Country</th>
              <th style="padding:10px 12px;text-align:left;">Action Required</th>
            </tr>
          </thead>
          <tbody>{rows_html}</tbody>
        </table>
        <p style="margin:20px 0 0;font-size:12px;color:#9ca3af;">Automated alert from CyberScan Pro. Full report attached.</p>
      </div>
    </div>
    </body></html>
    '''


def send_alert_email(
    sender:     str,
    password:   str,
    recipient:  str,
    df:         pd.DataFrame,
    scan_time:  str,
    attach_pdf: bool = True
) -> bool:
    if df is None or len(df) == 0:
        print('Email error: no scan data.')
        return False

    try:
        crit_cnt = int((df['severity'] == 'Critical').sum())
        high_cnt = int((df['severity'] == 'High').sum())

        msg = MIMEMultipart('mixed')
        msg['Subject'] = (
            'CyberScan Alert - {} findings ({} Critical, {} High) - {}'.format(
                len(df), crit_cnt, high_cnt, scan_time
            )
        )
        msg['From'] = sender
        msg['To']   = recipient

        # ✅ HTML body attached directly to mixed — no nested alternative wrapper
        msg.attach(MIMEText(_build_html_body(df, scan_time), 'html'))

        # ✅ PDF attachment
        if attach_pdf:
            try:
                ts       = str(scan_time).replace(' ', '_').replace(':', '-').replace('/', '-')
                pdf_path = os.path.join(REPORTS_DIR, 'scan_report_{}.pdf'.format(ts))
                generate_pdf_report(df, scan_time, pdf_path)
                with open(pdf_path, 'rb') as f:
                    pdf_part = MIMEApplication(f.read(), _subtype='pdf')
                    pdf_part.add_header(
                        'Content-Disposition', 'attachment',
                        filename=os.path.basename(pdf_path)
                    )
                    msg.attach(pdf_part)
                print('PDF attached successfully')
            except Exception as pdf_err:
                print('PDF generation failed, sending without PDF: {}'.format(pdf_err))

        # ✅ SMTP sending is OUTSIDE the if attach_pdf block
        try:
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(sender, password)
                server.sendmail(sender, recipient, msg.as_string())
        except smtplib.SMTPConnectError:
            with smtplib.SMTP('smtp.gmail.com', 587) as server:
                server.starttls()
                server.login(sender, password)
                server.sendmail(sender, recipient, msg.as_string())

        print('Email sent to {}'.format(recipient))
        return True

    except Exception as e:
        print('Email error: {}'.format(e))
        import traceback
        traceback.print_exc()
        return False

Overwriting modules/emailer.py


In [7]:
open('modules/__init__.py', 'w').close()
print('modules/__init__.py ✅')  # is just an empty file — it's just a marker that tells Python "this folder is a package". No code needed inside it.

modules/__init__.py ✅


In [8]:
import os

for folder in ['dashboard', 'dashboard/pages', '.streamlit']:
    os.makedirs(folder, exist_ok=True)
    print(f'  ✅  {folder}/')

print('Folders created ✅')  #  just checking is that there or not 

  ✅  dashboard/
  ✅  dashboard/pages/
  ✅  .streamlit/
Folders created ✅


 #API code 

In [9]:
%%writefile api.py
from fastapi import FastAPI, HTTPException, Depends, Header, Request
from pydantic import BaseModel
from typing import List, Optional
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded
import os

CYBERSCAN_API_KEY = os.environ.get("CYBERSCAN_API_KEY", "dev-key")

limiter = Limiter(key_func=get_remote_address)
app = FastAPI(
    title="CyberScan Pro API",
    version="4.0",
    description="Authenticated REST API — full enriched scan results with aggregated analysis",
)
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

SCAN_DATA: List[dict] = []


class ScanRecord(BaseModel):
    ip:                str
    port:              str
    protocol:          Optional[str]  = "tcp"
    state:             Optional[str]  = "open"
    service:           str
    product:           Optional[str]  = ""
    version:           Optional[str]  = ""
    malicious_reports: int            = 0
    suspicious_count:  int            = 0
    harmless_count:    int            = 0
    community_score:   int            = 0
    country:           Optional[str]  = "Unknown"
    network:           Optional[str]  = "Unknown"
    categories:        Optional[str]  = ""
    exposure_score:    float          = 0.0
    threat_score:      float          = 0.0
    context_score:     float          = 0.0
    risk_score:        float          = 0.0
    severity:          str            = "Low"
    recommendation:    Optional[str]  = ""


def verify_key(x_api_key: str = Header(..., description="CyberScan API key")):
    if x_api_key != CYBERSCAN_API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API key")


@app.get("/")
def root():
    return {"status": "running", "version": "4.0", "records": len(SCAN_DATA)}


@app.post("/load", dependencies=[Depends(verify_key)])
@limiter.limit("20/minute")
def load_data(request: Request, records: List[ScanRecord]):
    global SCAN_DATA
    SCAN_DATA = [r.dict() for r in records]
    return {
        "status":  "loaded",
        "records": len(SCAN_DATA),
        "hosts":   len({r["ip"] for r in SCAN_DATA}),
    }


@app.get("/results", dependencies=[Depends(verify_key)])
@limiter.limit("60/minute")
def get_results(request: Request, severity: Optional[str] = None):
    if not SCAN_DATA:
        raise HTTPException(status_code=404, detail="No scan data loaded")
    data = SCAN_DATA
    if severity:
        data = [r for r in data if r["severity"].lower() == severity.lower()]
    return {"count": len(data), "results": data}


@app.get("/analysis", dependencies=[Depends(verify_key)])
@limiter.limit("30/minute")
def get_analysis(request: Request):
    if not SCAN_DATA:
        raise HTTPException(status_code=404, detail="No scan data loaded")
    sev_counts: dict = {}
    for r in SCAN_DATA:
        sev = r.get("severity", "Unknown")
        sev_counts[sev] = sev_counts.get(sev, 0) + 1
    scores = [r["risk_score"] for r in SCAN_DATA]
    return {
        "total_records":      len(SCAN_DATA),
        "total_hosts":        len({r["ip"] for r in SCAN_DATA}),
        "severity_breakdown": sev_counts,
        "max_risk_score":     max(scores),
        "avg_risk_score":     round(sum(scores) / len(scores), 2),
        "vt_flagged_ips":     len({r["ip"] for r in SCAN_DATA if r.get("malicious_reports", 0) > 0}),
    }


@app.get("/host/{ip}", dependencies=[Depends(verify_key)])
def get_host(ip: str, request: Request):
    host_records = [r for r in SCAN_DATA if r["ip"] == ip]
    if not host_records:
        raise HTTPException(status_code=404, detail=f"No records found for {ip}")
    return {"ip": ip, "count": len(host_records), "records": host_records}

Overwriting api.py


# Here Will be App.py Dashboard

In [10]:
%%writefile dashboard/app.py
import streamlit as st
import os
import sys
import pandas as pd
from datetime import datetime

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..'))

from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(__file__), '..', 'u.env'))

from modules.scanner  import run_nmap_scan, parse_nmap_xml, check_virustotal
from modules.analyser import enrich_dataframe, generate_summary, build_host_summary
from modules.database import save_scan
from modules.emailer  import send_alert_email

VT_KEY       = os.environ.get('VT_API_KEY', '')
API_KEY      = os.environ.get('CYBERSCAN_API_KEY', 'dev-key')
TARGETS_RAW  = os.environ.get('SCAN_TARGETS', 'scanme.nmap.org')
GMAIL_SENDER = os.environ.get('GMAIL_SENDER', '')
GMAIL_PASS   = os.environ.get('GMAIL_PASSWORD', '')
GMAIL_RCPT   = os.environ.get('GMAIL_RECIPIENT', '')
TARGETS      = [t.strip() for t in TARGETS_RAW.split(',') if t.strip()]

st.set_page_config(
    page_title='CyberScan Pro',
    page_icon='🛡️',
    layout='wide',
    initial_sidebar_state='expanded'
)

for key, val in [
    ('df', None), ('scan_time', None), ('host_sum', None),
    ('last_scan_id', None), ('scan_running', False),
    ('targets_override', None),
]:
    if key not in st.session_state:
        st.session_state[key] = val

with st.sidebar:
    st.title('🛡️ CyberScan Pro')
    st.divider()

    st.subheader('⚙️ Status')
    if VT_KEY:
        st.success('VT API key ready ✅')
    else:
        st.error('VT_API_KEY not set ❌')

    if GMAIL_SENDER and GMAIL_PASS:
        st.success('Email configured ✅')
    else:
        st.warning('Email not configured ⚠️')

    st.divider()

    st.subheader('🎯 Scan Targets')
    targets_input = st.text_area(
        'Enter targets (one per line or comma separated)',
        value='\n'.join(
            st.session_state.targets_override
            if st.session_state.targets_override
            else TARGETS
        ),
        height=100,
        help='e.g. scanme.nmap.org or 192.168.1.1'
    )
    if st.button('💾 Save Targets', use_container_width=True):
        new_targets = [
            t.strip()
            for line in targets_input.splitlines()
            for t in line.split(',')
            if t.strip()
        ]
        st.session_state.targets_override = new_targets
        st.success(f'✅ Targets saved: {new_targets}')

    st.divider()

    scan_btn    = st.button('🚀 Run Full Scan', use_container_width=True, type='primary')
    refresh_btn = st.button('🔄 Clear Results', use_container_width=True)

    st.divider()
    if st.session_state.scan_time:
        st.caption(f'Last scan: {st.session_state.scan_time}')

st.title('🛡️ CyberScan Pro')
st.caption('Professional Network Reconnaissance & Threat Intelligence Platform')
st.divider()

if refresh_btn:
    for key in ['df', 'scan_time', 'host_sum', 'last_scan_id']:
        st.session_state[key] = None
    st.rerun()

if scan_btn:
    if not VT_KEY:
        st.error('VT_API_KEY is not set. Add it to your u.env file.')
        st.stop()

    active_targets = st.session_state.targets_override or TARGETS

    bar    = st.progress(0, text='Starting scan...')
    status = st.empty()

    all_rows = []
    for i, target in enumerate(active_targets):
        status.info(f'🔍 Scanning {target}  ({i+1}/{len(active_targets)})')
        xml = run_nmap_scan(target)
        all_rows.extend(parse_nmap_xml(xml))
        bar.progress(int((i + 1) / len(active_targets) * 40))

    if not all_rows:
        st.warning('Nmap returned no results. Check targets and network connectivity.')
        st.stop()

    status.info('🌐 Querying VirusTotal...')
    df_raw     = pd.DataFrame(all_rows)
    unique_ips = df_raw['ip'].unique().tolist()
    vt_data    = {}
    for j, ip in enumerate(unique_ips):
        vt_data[ip] = check_virustotal(ip, VT_KEY)
        bar.progress(40 + int((j + 1) / len(unique_ips) * 30))

    status.info('📊 Running analysis engine...')
    df       = enrich_dataframe(df_raw, vt_data)
    host_sum = build_host_summary(df)
    bar.progress(80)

    status.info('💾 Saving to database...')
    scan_id = save_scan(df, active_targets)
    bar.progress(90)

    scan_time_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # ── Auto email ────────────────────────────────────────────────────────────
    # ✅ FIXED: keyword args so df and scan_time land in the right parameters
    if GMAIL_SENDER and GMAIL_PASS and GMAIL_RCPT:
        status.info('📧 Sending alert email...')
        try:
            sent = send_alert_email(
                GMAIL_SENDER,
                GMAIL_PASS,
                GMAIL_RCPT,
                df=df,
                scan_time=scan_time_str,
                attach_pdf=True
            )
            if sent:
                st.success(f'✅ Alert email sent to {GMAIL_RCPT}!')
            else:
                st.warning('⚠️ Email returned False — check logs.')
        except Exception as e:
            st.warning(f'⚠️ Email failed: {e}')
    else:
        st.warning('⚠️ Email not configured — skipping alert.')

    bar.progress(100, text='Scan complete!')
    status.empty()

    st.session_state.df           = df
    st.session_state.host_sum     = host_sum
    st.session_state.scan_time    = scan_time_str
    st.session_state.last_scan_id = scan_id
    st.rerun()

df       = st.session_state.df
host_sum = st.session_state.host_sum

if df is None:
    st.info('🕐 No scan data yet. Click **Run Full Scan** in the sidebar.')
    st.stop()

summary = generate_summary(df, host_sum)

st.markdown(
    f'<div style="background:{summary["colour"]};padding:16px;border-radius:6px;'
    f'text-align:center;margin-bottom:16px;">'
    f'<h2 style="color:white;margin:0;">Overall Security Posture: {summary["posture"]}</h2>'
    f'</div>',
    unsafe_allow_html=True
)

c1, c2, c3, c4, c5, c6 = st.columns(6)
c1.metric('🖥️ Hosts',     summary['total_hosts'])
c2.metric('🔓 Open Ports', summary['total_ports'])
c3.metric('🚨 Critical',   summary['crit_hosts'])
c4.metric('⚠️ High',       summary['high_hosts'])
c5.metric('🦠 VT Flagged', summary['vt_flagged'])
c6.metric('📈 Max Risk',   f"{summary['max_risk']:.1f}")
st.divider()

st.subheader('⚡ Key Findings')
for finding in summary['findings']:
    if 'No critical' in finding:
        st.success(finding)
    elif 'Critical' in finding or 'malicious' in finding:
        st.error(finding)
    else:
        st.warning(finding)

st.divider()

st.subheader('🖥️ Host Overview')
sev_order    = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3}
host_display = host_sum.copy()
host_display['sort'] = host_display['overall_severity'].map(sev_order)
host_display = host_display.sort_values('sort').drop('sort', axis=1)
st.dataframe(
    host_display[['ip', 'open_ports', 'max_risk', 'overall_severity',
                  'country', 'categories', 'services']],
    use_container_width=True,
    hide_index=True,
)


Overwriting dashboard/app.py


In [11]:
%%writefile dashboard/pages/1_Overview.py
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
import os, sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))
from modules.analyser import build_host_summary, generate_summary

st.title('🛡️ CyberScan Pro — Overview')

df = st.session_state.get('df')
if df is None:
    st.info('No scan data yet. Run a scan on the main page.')
    st.stop()

host_sum = st.session_state.get('host_sum')
if host_sum is None:
    host_sum = build_host_summary(df)

summary = generate_summary(df, host_sum)

CRIT   = '#ef4444'
HIGH   = '#f97316'
MED    = '#eab308'
LOW    = '#22c55e'
ACCENT = '#4f8ef7'
SEV_COLORS = {'Critical': CRIT, 'High': HIGH, 'Medium': MED, 'Low': LOW}

posture_bg = {'CRITICAL': CRIT, 'HIGH RISK': HIGH, 'MODERATE': MED, 'LOW RISK': LOW}
posture_col = posture_bg.get(summary['posture'], ACCENT)

st.markdown(
    f'<div style="background:{posture_col};padding:20px;border-radius:8px;'
    f'text-align:center;margin-bottom:16px;">'
    f'<h2 style="color:white;margin:0;">🛡️ CYBERSCAN PRO</h2>'
    f'<p style="color:rgba(255,255,255,0.8);margin:6px 0 0;">'
    f'Security Posture: <strong>{summary["posture"]}</strong> | '
    f'Max Risk: {summary["max_risk"]:.1f}/10</p>'
    f'</div>',
    unsafe_allow_html=True
)

c1,c2,c3,c4,c5,c6 = st.columns(6)
c1.metric('🖥️ Hosts',     summary['total_hosts'])
c2.metric('🔓 Ports',      summary['total_ports'])
c3.metric('🚨 Critical',   summary['crit_hosts'])
c4.metric('⚠️ High',       summary['high_hosts'])
c5.metric('🦠 VT Flagged', summary['vt_flagged'])
c6.metric('📈 Max Risk',   f"{summary['max_risk']:.1f}")
st.divider()

# ── Key Findings ──────────────────────────────────────────────────────────────
st.subheader('⚡ Key Findings')
for finding in summary['findings']:
    if 'No critical' in finding:
        st.success(finding)
    elif 'malicious' in finding or 'Critical' in finding:
        st.error(finding)
    else:
        st.warning(finding)
st.divider()

# ── Row 1: Severity bar + pie ─────────────────────────────────────────────────
sev_counts = df['severity'].value_counts().reindex(
    ['Critical','High','Medium','Low'], fill_value=0
)
col1, col2 = st.columns(2)

with col1:
    st.subheader('📊 Findings by Severity')
    fig = go.Figure(go.Bar(
        x=sev_counts.index,
        y=sev_counts.values,
        marker=dict(color=[SEV_COLORS[s] for s in sev_counts.index]),
        text=sev_counts.values,
        textposition='outside',
    ))
    fig.update_layout(template='plotly_dark', height=320,
                      margin=dict(t=30,b=20,l=20,r=20))
    st.plotly_chart(fig, use_container_width=True)

with col2:
    st.subheader('🥧 Severity Distribution')
    fig2 = go.Figure(go.Pie(
        labels=sev_counts.index,
        values=sev_counts.values,
        hole=0.55,
        marker=dict(colors=[SEV_COLORS[s] for s in sev_counts.index]),
    ))
    fig2.update_layout(
        template='plotly_dark', height=320,
        margin=dict(t=30,b=20,l=20,r=20),
        annotations=[dict(text='RISK', x=0.5, y=0.5,
                          font_size=18, showarrow=False)]
    )
    st.plotly_chart(fig2, use_container_width=True)

# ── Row 2: Country bar + Radar ────────────────────────────────────────────────
col3, col4 = st.columns(2)

with col3:
    st.subheader('🌍 Findings by Country')
    country_counts = df['country'].value_counts().head(8)
    fig3 = go.Figure(go.Bar(
        x=country_counts.values,
        y=country_counts.index,
        orientation='h',
        marker=dict(color=[
            CRIT if c in {'CN','RU','KP','IR','NG','UA','VN','RO'} else ACCENT
            for c in country_counts.index
        ]),
        text=country_counts.values,
        textposition='outside',
    ))
    fig3.update_layout(template='plotly_dark', height=340,
                       margin=dict(t=30,b=20,l=80,r=40))
    st.plotly_chart(fig3, use_container_width=True)

with col4:
    st.subheader('🎯 Score Radar')
    score_means = df[['exposure_score','threat_score',
                      'context_score','risk_score']].mean()
    fig4 = go.Figure(go.Scatterpolar(
        r=score_means.values.tolist() + [score_means.values[0]],
        theta=['Exposure','Threat','Context','Risk','Exposure'],
        fill='toself',
        fillcolor='rgba(79,142,247,0.25)',
        line=dict(color=ACCENT, width=2),
    ))
    fig4.update_layout(
        polar=dict(
            bgcolor='rgba(0,0,0,0)',
            angularaxis=dict(color='white'),
            radialaxis=dict(color='white', range=[0,10])
        ),
        template='plotly_dark', height=340,
        margin=dict(t=30,b=20,l=20,r=20)
    )
    st.plotly_chart(fig4, use_container_width=True)

# ── Row 3: Risk score histogram ───────────────────────────────────────────────
st.subheader('📈 Risk Score Distribution')
fig5 = px.histogram(
    df, x='risk_score', nbins=20,
    color='severity',
    color_discrete_map=SEV_COLORS,
    title='Distribution of Risk Scores Across All Findings',
    template='plotly_dark',
    labels={'risk_score': 'Risk Score (0-10)'}
)
fig5.update_layout(height=320, margin=dict(t=50,b=20,l=40,r=20))
st.plotly_chart(fig5, use_container_width=True)

# ── Row 4: Service risk treemap ───────────────────────────────────────────────
st.subheader('🗺️ Service Risk Treemap')
svc_data = df.groupby(['service','severity']).size().reset_index(name='count')
fig6 = px.treemap(
    svc_data,
    path=['severity','service'],
    values='count',
    color='severity',
    color_discrete_map=SEV_COLORS,
    template='plotly_dark',
)
fig6.update_layout(height=400, margin=dict(t=30,b=20,l=20,r=20))
st.plotly_chart(fig6, use_container_width=True)

# ── Row 5: Host risk gauge ────────────────────────────────────────────────────
st.subheader('🎰 Host Risk Gauges')
cols = st.columns(min(len(host_sum), 4))
for i, (_, row) in enumerate(host_sum.head(4).iterrows()):
    with cols[i]:
        gauge = go.Figure(go.Indicator(
            mode='gauge+number',
            value=float(row['max_risk']),
            title={'text': str(row['ip']), 'font': {'size': 11}},
            gauge=dict(
                axis=dict(range=[0, 10]),
                bar=dict(color=CRIT if row['max_risk'] >= 7
                         else HIGH if row['max_risk'] >= 5
                         else MED if row['max_risk'] >= 3
                         else LOW),
                steps=[
                    dict(range=[0, 3],   color='rgba(34,197,94,0.15)'),
                    dict(range=[3, 5],   color='rgba(234,179,8,0.15)'),
                    dict(range=[5, 7],   color='rgba(249,115,22,0.15)'),
                    dict(range=[7, 10],  color='rgba(239,68,68,0.15)'),
                ],
                threshold=dict(
                    line=dict(color='white', width=2),
                    thickness=0.75,
                    value=row['max_risk']
                )
            )
        ))
        gauge.update_layout(
            template='plotly_dark',
            height=220,
            margin=dict(t=40,b=10,l=20,r=20)
        )
        st.plotly_chart(gauge, use_container_width=True)

Overwriting dashboard/pages/1_Overview.py


In [12]:
%%writefile dashboard/pages/2_Analysis.py
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
import os, sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))
from modules.analyser import build_host_summary

st.title('🔍 Deep Analysis')

df = st.session_state.get('df')
if df is None:
    st.info('No scan data yet. Run a scan on the main page.')
    st.stop()

host_sum = st.session_state.get('host_sum')
if host_sum is None:
    host_sum = build_host_summary(df)

BG     = '#0f1117'
CRIT   = '#ef4444'
HIGH   = '#f97316'
MED    = '#eab308'
LOW    = '#22c55e'
ACCENT = '#4f8ef7'
SEV_COLORS = {'Critical': CRIT, 'High': HIGH, 'Medium': MED, 'Low': LOW}

st.subheader('🎯 Exposure vs Threat Intelligence')
scatter = px.scatter(
    df,
    x='exposure_score', y='threat_score',
    size='risk_score',
    color='severity',
    hover_data=['ip','port','service','country','risk_score'],
    color_discrete_map=SEV_COLORS,
    title='Exposure vs Threat Intelligence (bubble size = overall risk)',
    labels={'exposure_score':'Exposure Score (0-10)','threat_score':'Threat Score (0-10)'},
    template='plotly_dark'
)
scatter.update_layout(
    height=450,
    margin=dict(t=60,b=40,l=40,r=20),
    xaxis=dict(range=[-0.5,11]),
    yaxis=dict(range=[-0.5,11]),
)
scatter.update_traces(marker=dict(line=dict(color=BG, width=1.5)))
st.plotly_chart(scatter, use_container_width=True)

st.subheader('🔧 Risk Score by Service')
svc_risk = df.groupby('service').agg(
    avg_risk=('risk_score','mean'),
    count=('port','count'),
    max_risk=('risk_score','max')
).reset_index().sort_values('avg_risk', ascending=False).head(12)

svc_bar = go.Figure()
svc_bar.add_trace(go.Bar(
    name='Avg Risk',
    x=svc_risk['service'], y=svc_risk['avg_risk'],
    marker_color=ACCENT, opacity=0.9
))
svc_bar.add_trace(go.Bar(
    name='Max Risk',
    x=svc_risk['service'], y=svc_risk['max_risk'],
    marker_color=CRIT, opacity=0.7
))
svc_bar.update_layout(
    barmode='group',
    template='plotly_dark', height=360,
    margin=dict(t=60,b=40,l=40,r=20),
    yaxis=dict(range=[0,11]),
)
st.plotly_chart(svc_bar, use_container_width=True)

st.subheader('🔓 Open Port Frequency')
port_counts = df['port'].value_counts().head(15).sort_values()
port_bar = go.Figure(go.Bar(
    x=port_counts.values, y=port_counts.index.astype(str),
    orientation='h',
    marker=dict(
        color=port_counts.values,
        colorscale=[[0,'#22c55e'],[0.5,'#eab308'],[1,'#ef4444']],
    ),
    text=port_counts.values, textposition='outside',
))
port_bar.update_layout(
    template='plotly_dark', height=420,
    margin=dict(t=60,b=20,l=60,r=40),
)
st.plotly_chart(port_bar, use_container_width=True)

st.subheader('🦠 VirusTotal Engine Reports per Host')
vt_df = df.groupby('ip').agg(
    malicious=('malicious_reports','max'),
    suspicious=('suspicious_count','max'),
    harmless=('harmless_count','max'),
).reset_index()

vt_bar = go.Figure()
for col, colour, label in [
    ('malicious',  CRIT, 'Malicious'),
    ('suspicious', HIGH, 'Suspicious'),
    ('harmless',   LOW,  'Harmless'),
]:
    vt_bar.add_trace(go.Bar(
        name=label, x=vt_df['ip'], y=vt_df[col],
        marker_color=colour, opacity=0.85
    ))
vt_bar.update_layout(
    barmode='group',
    template='plotly_dark', height=380,
    margin=dict(t=60,b=40,l=40,r=20),
)
st.plotly_chart(vt_bar, use_container_width=True)

st.subheader('🌡️ Host Risk Heatmap')
host_heatmap_df = host_sum.set_index('ip')[['max_risk','max_exposure','max_threat']].T
heatmap = go.Figure(go.Heatmap(
    z=host_heatmap_df.values,
    x=host_heatmap_df.columns.tolist(),
    y=['Max Risk','Max Exposure','Max Threat'],
    colorscale=[
        [0.0, '#16a34a'],
        [0.3, '#eab308'],
        [0.6, '#f97316'],
        [1.0, '#ef4444'],
    ],
    zmin=0, zmax=10,
    text=[[f'{v:.1f}' for v in row] for row in host_heatmap_df.values],
    texttemplate='%{text}',
    textfont=dict(size=13, color='white'),
))
heatmap.update_layout(
    template='plotly_dark', height=280,
    margin=dict(t=60,b=40,l=120,r=20),
)
st.plotly_chart(heatmap, use_container_width=True)

st.subheader('🏆 Top 10 Highest Risk Findings')
top_risks = df.sort_values('risk_score', ascending=True).tail(10)
top_bar = go.Figure(go.Bar(
    x=top_risks['risk_score'],
    y=[f"{r['ip']}:{r['port']} ({r['service']})" for _, r in top_risks.iterrows()],
    orientation='h',
    marker=dict(
        color=top_risks['risk_score'],
        colorscale=[[0,'#22c55e'],[0.5,'#f97316'],[1,'#ef4444']],
    ),
    text=[f"{v:.1f}" for v in top_risks['risk_score']],
    textposition='outside',
))
top_bar.update_layout(
    template='plotly_dark', height=420,
    xaxis=dict(range=[0,11]),
    margin=dict(t=60,b=20,l=200,r=60),
)
st.plotly_chart(top_bar, use_container_width=True)

Overwriting dashboard/pages/2_Analysis.py


In [30]:
%%writefile dashboard/pages/3_History.py
import streamlit as st
import plotly.graph_objects as go
import plotly.express as px
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))
from modules.database import load_history, load_scan_by_id

st.title('📜 Scan History')
st.caption('Historical trend data stored in SQLite')

history_df = load_history()
if history_df.empty:
    st.info('No scan history yet. Run at least one scan.')
    st.stop()

# ── Sort by scan id so charts are in correct order ────────────────────────────
history_df = history_df.sort_values('id').reset_index(drop=True)

CRIT   = '#ef4444'
HIGH   = '#f97316'
ACCENT = '#4f8ef7'

# ── All Scans Table with progress bars ───────────────────────────────────────
st.subheader('📋 All Scans')
st.dataframe(
    history_df,
    use_container_width=True,
    hide_index=True,
    column_config={
        'max_risk_score': st.column_config.ProgressColumn(
            'Max Risk', min_value=0, max_value=10, format='%.2f'
        ),
        'avg_risk_score': st.column_config.ProgressColumn(
            'Avg Risk', min_value=0, max_value=10, format='%.2f'
        ),
        'critical_count': st.column_config.NumberColumn('Critical'),
        'high_count':     st.column_config.NumberColumn('High'),
        'total_hosts':    st.column_config.NumberColumn('Hosts'),
        'total_ports':    st.column_config.NumberColumn('Ports'),
    }
)

st.divider()

# ── Risk Score Evolution ──────────────────────────────────────────────────────
if len(history_df) > 1:
    st.subheader('📈 Risk Score Evolution Over Time')
    risk_line = go.Figure()
    risk_line.add_trace(go.Scatter(
        x=history_df['scan_time'],
        y=history_df['avg_risk_score'],
        mode='lines+markers',
        line=dict(color=ACCENT, width=2),
        marker=dict(size=8, color=ACCENT),
        name='Avg Risk Score'
    ))
    risk_line.add_trace(go.Scatter(
        x=history_df['scan_time'],
        y=history_df['max_risk_score'],
        mode='lines+markers',
        line=dict(color=CRIT, width=2, dash='dash'),
        marker=dict(size=8, color=CRIT),
        name='Max Risk Score'
    ))
    risk_line.update_layout(
        template='plotly_dark', height=400,
        xaxis=dict(title='Scan Time'),          # ✅ No broken range=[0,11]
        yaxis=dict(title='Risk Score (0–10)', range=[0, 11]),
        margin=dict(t=60, b=40, l=60, r=40),
    )
    st.plotly_chart(risk_line, use_container_width=True)
else:
    st.info('Run at least 2 scans to see the Risk Score Evolution chart.')

st.divider()


# ── Hosts vs Ports Bubble Scatter ─────────────────────────────────────────────
st.subheader('🔵 Hosts vs Ports Across Scans')
hosts_ports = px.scatter(
    history_df,
    x='total_hosts', y='total_ports',
    size='avg_risk_score',
    color='max_risk_score',
    color_continuous_scale=['#22c55e', '#eab308', '#ef4444'],
    hover_data=['targets', 'scan_time'],
    title='Hosts vs Ports Across Scans',
    template='plotly_dark'
)
hosts_ports.update_layout(
    height=400,
    margin=dict(t=60, b=40, l=60, r=40),
)
st.plotly_chart(hosts_ports, use_container_width=True)

st.divider()

# ── Drill Into a Past Scan ────────────────────────────────────────────────────
st.subheader('🔎 Drill Into a Past Scan')
selected_id = st.selectbox('Select scan ID', history_df['id'].tolist())
if selected_id:
    past_df = load_scan_by_id(selected_id)
    if not past_df.empty:
        st.caption(f'Scan {selected_id}: {len(past_df)} rows, {past_df["ip"].nunique()} hosts')
        show_cols = [c for c in [
            'ip', 'port', 'service', 'risk_score',
            'severity', 'country', 'recommendation'
        ] if c in past_df.columns]
        st.dataframe(
            past_df[show_cols].sort_values('risk_score', ascending=False),
            use_container_width=True,
            hide_index=True
        )

Overwriting dashboard/pages/3_History.py


In [14]:
%%writefile dashboard/pages/4_Settings.py
import streamlit as st
import os, sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))

from dotenv import load_dotenv
load_dotenv('u.env')

st.title('⚙️ Session Configuration')
st.caption('Update your API keys, scan targets, and email settings')

st.divider()

# ── API Credentials ───────────────────────────────────────────────────────────
st.subheader('🔑 API Credentials')

vt_key = st.text_input(
    'VirusTotal API Key',
    value=os.environ.get('VT_API_KEY', ''),
    type='password',
    placeholder='Enter VirusTotal API key'
)
api_key = st.text_input(
    'CyberScan API Key',
    value=os.environ.get('CYBERSCAN_API_KEY', 'dev-key'),
    placeholder='dev-key'
)

st.divider()

# ── Scan Targets ──────────────────────────────────────────────────────────────
st.subheader('🎯 Scan Targets')

targets_raw = st.text_input(
    'Scan Targets (comma separated)',
    value=os.environ.get('SCAN_TARGETS', 'scanme.nmap.org'),
    placeholder='host1,host2,192.168.1.1'
)

st.divider()

# ── Email Alerts ──────────────────────────────────────────────────────────────
st.subheader('📧 Email Alerts')

gmail_sender = st.text_input(
    'Gmail Sender',
    value=os.environ.get('GMAIL_SENDER', ''),
    placeholder='youremail@gmail.com'
)
gmail_pass = st.text_input(
    'Gmail App Password',
    value=os.environ.get('GMAIL_PASSWORD', ''),
    type='password',
    placeholder='16-char app password'
)
gmail_rcpt = st.text_input(
    'Alert Recipient',
    value=os.environ.get('GMAIL_RECIPIENT', ''),
    placeholder='recipient@company.com'
)

st.divider()

# ── Apply Button ──────────────────────────────────────────────────────────────
if st.button('💾 Apply Settings', type='primary', use_container_width=True):
    targets = [t.strip() for t in targets_raw.split(',') if t.strip()]

    st.session_state['vt_key']      = vt_key.strip()
    st.session_state['api_key']     = api_key.strip()
    st.session_state['targets']     = targets
    st.session_state['targets_raw'] = targets_raw.strip()
    st.session_state['gmail_sender']= gmail_sender.strip()
    st.session_state['gmail_pass']  = gmail_pass.strip()
    st.session_state['gmail_rcpt']  = gmail_rcpt.strip()

    st.success('Settings applied for this session ✅')
    st.write(f'VT Key:    {"✅ Set" if vt_key else "❌ Empty"}')
    st.write(f'Targets:   {targets}')
    st.write(f'From:      {gmail_sender or "❌ Empty"}')
    st.write(f'Recipient: {gmail_rcpt   or "❌ Empty"}')
    st.write(f'Password:  {"✅ Set" if gmail_pass else "❌ Empty"}')
    st.info('Re-run the scan on the main page to use new settings.')

st.divider()
st.caption('💡 Generate Gmail App Password at: myaccount.google.com → Security → App passwords')

Overwriting dashboard/pages/4_Settings.py


In [15]:
%%writefile dashboard/pages/5_Scan_Data.py
import streamlit as st
import pandas as pd
import os, sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))

st.title('📋 All Findings')
st.caption('Full port-level results sorted by Risk Score')

df = st.session_state.get('df')
if df is None:
    st.info('No scan data yet. Run a scan on the main page.')
    st.stop()

# ── Filters ───────────────────────────────────────────────────────────────────
col1, col2, col3 = st.columns(3)
with col1:
    sev_filter = st.multiselect(
        'Filter by Severity',
        ['Critical','High','Medium','Low'],
        default=['Critical','High','Medium','Low']
    )
with col2:
    svc_filter = st.multiselect(
        'Filter by Service',
        sorted(df['service'].unique().tolist()),
        default=[]
    )
with col3:
    ip_search = st.text_input('Search IP', placeholder='e.g. 10.0.0')

# ── Apply filters ─────────────────────────────────────────────────────────────
filtered = df[df['severity'].isin(sev_filter)]
if svc_filter:
    filtered = filtered[filtered['service'].isin(svc_filter)]
if ip_search:
    filtered = filtered[filtered['ip'].str.contains(ip_search, na=False)]

st.caption(f'Showing {len(filtered)} of {len(df)} rows')

# ── Display columns ───────────────────────────────────────────────────────────
display_cols = [
    'ip','port','service','state','severity','risk_score',
    'exposure_score','threat_score','malicious_reports',
    'country','product','version','recommendation'
]
cols_to_show = [c for c in display_cols if c in filtered.columns]
table_df     = filtered[cols_to_show].sort_values('risk_score', ascending=False).reset_index(drop=True)

# ── Colour map for severity ───────────────────────────────────────────────────
def color_severity(val):
    colors = {
        'Critical': 'background-color: rgba(239,68,68,0.18); color: #ef4444',
        'High':     'background-color: rgba(249,115,22,0.18); color: #f97316',
        'Medium':   'background-color: rgba(234,179,8,0.18); color: #eab308',
        'Low':      'background-color: rgba(34,197,94,0.18); color: #22c55e',
    }
    return colors.get(val, '')

styled = (
    table_df.style
    .applymap(color_severity, subset=['severity'])
    .background_gradient(subset=['risk_score'], cmap='RdYlGn_r', vmin=0, vmax=10)
    .format({
        'risk_score':     '{:.2f}',
        'exposure_score': '{:.2f}',
        'threat_score':   '{:.2f}'
    })
    .hide(axis='index')
)

st.dataframe(
    table_df,
    use_container_width=True,
    hide_index=True,
    column_config={
        'risk_score':     st.column_config.ProgressColumn('Risk Score', min_value=0, max_value=10, format='%.2f'),
        'exposure_score': st.column_config.ProgressColumn('Exposure',   min_value=0, max_value=10, format='%.2f'),
        'threat_score':   st.column_config.ProgressColumn('Threat',     min_value=0, max_value=10, format='%.2f'),
        'severity':       st.column_config.TextColumn('Severity'),
    }
)

Overwriting dashboard/pages/5_Scan_Data.py


In [16]:
%%writefile dashboard/pages/6_Host_Summary.py
import streamlit as st
import pandas as pd
import os, sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))
from modules.analyser import build_host_summary

st.title('🖥️ Host-level Summary')
st.caption('One row per host — sorted by maximum risk score')

df = st.session_state.get('df')
if df is None:
    st.info('No scan data yet. Run a scan on the main page.')
    st.stop()

# ✅ FIX: use 'is None' instead of 'or' with DataFrame
host_sum = st.session_state.get('host_sum')
if host_sum is None:
    host_sum = build_host_summary(df)

host_display = host_sum[['ip','open_ports','max_risk','overall_severity',
                          'country','categories','services']].copy()

def color_severity(val):
    colors = {
        'Critical': 'background-color: rgba(239,68,68,0.18); color: #ef4444',
        'High':     'background-color: rgba(249,115,22,0.18); color: #f97316',
        'Medium':   'background-color: rgba(234,179,8,0.18); color: #eab308',
        'Low':      'background-color: rgba(34,197,94,0.18); color: #22c55e',
    }
    return colors.get(val, '')

st.dataframe(
    host_display.sort_values('max_risk', ascending=False),
    use_container_width=True,
    hide_index=True,
    column_config={
        'max_risk': st.column_config.ProgressColumn(
            'Max Risk', min_value=0, max_value=10, format='%.2f'
        ),
        'overall_severity': st.column_config.TextColumn('Severity'),
        'open_ports':       st.column_config.NumberColumn('Open Ports'),
        'ip':               st.column_config.TextColumn('IP Address'),
        'country':          st.column_config.TextColumn('Country'),
        'categories':       st.column_config.TextColumn('Categories'),
        'services':         st.column_config.TextColumn('Services'),
    }
)

st.divider()

c1, c2, c3, c4 = st.columns(4)
c1.metric('Total Hosts',    len(host_display))
c2.metric('Critical Hosts', int((host_display['overall_severity'] == 'Critical').sum()))
c3.metric('High Hosts',     int((host_display['overall_severity'] == 'High').sum()))
c4.metric('Max Risk',       f"{host_display['max_risk'].max():.2f}")

Overwriting dashboard/pages/6_Host_Summary.py


In [17]:
%%writefile dashboard/pages/7_Scan_History.py
import streamlit as st
import plotly.graph_objects as go
import os, sys

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))
from modules.database import load_history, load_scan_by_id

st.title('📜 Scan History')
st.caption('All past scans stored in SQLite database')

history = load_history()

if history.empty:
    st.info('No scan history yet. Run at least one scan.')
    st.stop()

CRIT   = '#ef4444'
ACCENT = '#4f8ef7'

# ── All scans table ───────────────────────────────────────────────────────────
st.subheader('📋 All Scans')
st.dataframe(
    history,
    use_container_width=True,
    hide_index=True,
    column_config={
        'max_risk_score': st.column_config.ProgressColumn(
            'Max Risk', min_value=0, max_value=10, format='%.2f'
        ),
        'avg_risk_score': st.column_config.ProgressColumn(
            'Avg Risk', min_value=0, max_value=10, format='%.2f'
        ),
        'critical_count': st.column_config.NumberColumn('Critical'),
        'high_count':     st.column_config.NumberColumn('High'),
        'total_hosts':    st.column_config.NumberColumn('Hosts'),
        'total_ports':    st.column_config.NumberColumn('Ports'),
    }
)

st.divider()

# ── Trend chart ───────────────────────────────────────────────────────────────
if len(history) > 1:
    st.subheader('📈 Risk Score Trend Over Time')
    hist_sorted = history.sort_values('id')
    trend = go.Figure()
    trend.add_trace(go.Scatter(
        x=hist_sorted['scan_time'], y=hist_sorted['max_risk_score'],
        name='Max Risk', mode='lines+markers',
        line=dict(color=CRIT, width=2),
        marker=dict(size=8, color=CRIT)
    ))
    trend.add_trace(go.Scatter(
        x=hist_sorted['scan_time'], y=hist_sorted['avg_risk_score'],
        name='Avg Risk', mode='lines+markers',
        line=dict(color=ACCENT, width=2),
        marker=dict(size=8, color=ACCENT)
    ))
    trend.update_layout(
        template='plotly_dark', height=340,
        margin=dict(t=60,b=40,l=40,r=20),
    )
    st.plotly_chart(trend, use_container_width=True)
else:
    st.info('Run more scans to see the trend chart.')

st.divider()

# ── Drill into past scan ──────────────────────────────────────────────────────
st.subheader('🔎 Drill Into a Past Scan')
selected_id = st.selectbox('Select scan ID', history['id'].tolist())
if selected_id:
    past_df = load_scan_by_id(selected_id)
    if not past_df.empty:
        st.caption(f'Scan {selected_id}: {len(past_df)} rows, {past_df["ip"].nunique()} hosts')
        show_cols = [c for c in [
            'ip','port','service','risk_score',
            'severity','country','recommendation'
        ] if c in past_df.columns]
        st.dataframe(
            past_df[show_cols].sort_values('risk_score', ascending=False),
            use_container_width=True,
            hide_index=True
        )

Overwriting dashboard/pages/7_Scan_History.py


In [18]:
%%writefile dashboard/pages/8_Export.py
import streamlit as st
import pandas as pd
import os, sys
from datetime import datetime

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))
from modules.emailer import generate_pdf_report
from modules.analyser import build_host_summary

st.title('📥 Export Scan Results')
st.caption('Download PDF report, full CSV, and host summary CSV')

df = st.session_state.get('df')
if df is None:
    st.info('No scan data yet. Run a scan on the main page.')
    st.stop()

# ✅ FIX: use 'is None' instead of 'or' with DataFrame
host_sum = st.session_state.get('host_sum')
if host_sum is None:
    host_sum = build_host_summary(df)

scan_time = st.session_state.get('scan_time') or datetime.now().strftime('%Y-%m-%d %H:%M:%S')

os.makedirs('reports', exist_ok=True)

ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
pdf_path = f'reports/cyberscan_report_{ts}.pdf'
csv_path = f'reports/cyberscan_data_{ts}.csv'
host_csv = f'reports/cyberscan_hosts_{ts}.csv'

export_cols = [
    'ip','port','service','state','severity','risk_score',
    'exposure_score','threat_score','context_score',
    'malicious_reports','suspicious_count','country',
    'categories','product','version','recommendation'
]
export_df = df[[c for c in export_cols if c in df.columns]]

if st.button('🔄 Generate Export Files', type='primary', use_container_width=True):
    with st.spinner('Generating files...'):
        generate_pdf_report(df, scan_time, pdf_path)
        export_df.to_csv(csv_path, index=False)
        host_sum.to_csv(host_csv, index=False)
    st.success('Files generated ✅')

    st.divider()

    col1, col2, col3 = st.columns(3)

    with col1:
        with open(pdf_path, 'rb') as f:
            st.download_button(
                label='📄 Download PDF Report',
                data=f.read(),
                file_name=os.path.basename(pdf_path),
                mime='application/pdf',
                use_container_width=True
            )

    with col2:
        with open(csv_path, 'rb') as f:
            st.download_button(
                label='📊 Download Full CSV',
                data=f.read(),
                file_name=os.path.basename(csv_path),
                mime='text/csv',
                use_container_width=True
            )

    with col3:
        with open(host_csv, 'rb') as f:
            st.download_button(
                label='🖥️ Download Host CSV',
                data=f.read(),
                file_name=os.path.basename(host_csv),
                mime='text/csv',
                use_container_width=True
            )

    st.divider()

    st.caption(
        f'PDF: {os.path.getsize(pdf_path):,} bytes  |  '
        f'CSV: {os.path.getsize(csv_path):,} bytes  |  '
        f'Rows: {len(export_df)}'
    )

Overwriting dashboard/pages/8_Export.py


In [19]:
%%writefile dashboard/pages/9_Email_Alert.py
import streamlit as st
import os
import sys
import traceback
from datetime import datetime

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', '..'))

from dotenv import load_dotenv
from pathlib import Path
_base = Path(__file__).resolve().parent.parent.parent
load_dotenv(_base / 'u.env', override=True)
load_dotenv('u.env', override=True)

from modules.emailer import send_alert_email

st.title('📧 Send Security Alert Email')
st.caption('Send a styled HTML report with optional PDF attachment to any recipient')

# ── Load env values ───────────────────────────────────────────────────────────
ENV_SENDER = os.environ.get('GMAIL_SENDER', '')
ENV_PASS   = os.environ.get('GMAIL_PASSWORD', '')
ENV_RCPT   = os.environ.get('GMAIL_RECIPIENT', '')

# ── Get scan data ─────────────────────────────────────────────────────────────
df        = st.session_state.get('df')
scan_time = st.session_state.get('scan_time', '')

# ── Guard: no scan data ───────────────────────────────────────────────────────
if df is None or len(df) == 0:
    st.error('❌ No scan data available. Please run a scan first.')
    st.info('👉 Go to the **Home page** and click **Run Full Scan**, then come back here.')
    if st.button('🔄 Refresh page', use_container_width=True):
        st.rerun()
    st.stop()

# ── Scan data summary banner ──────────────────────────────────────────────────
crit = int((df['severity'] == 'Critical').sum())
high = int((df['severity'] == 'High').sum())
med  = int((df['severity'] == 'Medium').sum())
low  = int((df['severity'] == 'Low').sum())

col1, col2, col3, col4, col5 = st.columns(5)
col1.metric('🚨 Critical', crit)
col2.metric('⚠️ High',     high)
col3.metric('🟡 Medium',   med)
col4.metric('🟢 Low',      low)
col5.metric('📋 Total',    len(df))

st.divider()

# ── Email settings ────────────────────────────────────────────────────────────
st.subheader('📧 Email Settings')

st.info(
    '💡 **Gmail App Password required.** Go to '
    '[myaccount.google.com → Security → App passwords]'
    '(https://myaccount.google.com/apppasswords) '
    'to generate a 16-character password.',
    icon=None
)

sender = st.text_input(
    'Sender Gmail address',
    value=ENV_SENDER,
    placeholder='youremail@gmail.com',
    help='Must be a Gmail account with 2-Step Verification enabled'
)

password = st.text_input(
    'Gmail App Password',
    value=ENV_PASS,
    type='password',
    placeholder='16-character app password (no spaces)',
    help='This is NOT your regular Gmail password. Generate one at myaccount.google.com → Security → App passwords'
)

recipient = st.text_input(
    'Recipient email',
    value=ENV_RCPT,
    placeholder='recipient@company.com',
    help='Who should receive this security report'
)

attach_pdf = st.checkbox('Attach PDF report', value=True, help='Attach a formatted PDF alongside the HTML email')

st.divider()

# ── Live validation ───────────────────────────────────────────────────────────
ready   = True
reasons = []

if not sender or not sender.strip():
    reasons.append('Sender email is empty')
    ready = False
elif '@gmail.com' not in sender:
    reasons.append('Sender must be a @gmail.com address')
    ready = False

if not password or not password.strip():
    reasons.append('App password is empty')
    ready = False
elif len(password.replace(' ', '').strip()) != 16:
    # ✅ FIX: strip spaces BEFORE checking length
    # Google shows app passwords as "xxxx xxxx xxxx xxxx" (19 chars with spaces)
    # but the actual password is 16 chars. We must remove spaces first.
    reasons.append(f'App password should be exactly 16 characters (yours is {len(password.replace(" ", "").strip())} after removing spaces)')
    ready = False

if not recipient or not recipient.strip():
    reasons.append('Recipient email is empty')
    ready = False
elif '@' not in recipient:
    reasons.append('Recipient email looks invalid')
    ready = False

if not ready:
    for r in reasons:
        st.warning(f'⚠️ {r}')
else:
    st.success('✅ All fields look good — ready to send!')

st.divider()

# ── Send button ───────────────────────────────────────────────────────────────
send_clicked = st.button(
    '📧 Send Alert Email',
    type='primary',
    use_container_width=True,
    disabled=not ready
)

if send_clicked:
    # Clean the app password: remove any spaces Google adds
    password_clean = password.replace(' ', '').strip()

    with st.spinner(f'Sending report to {recipient}... this may take 10–20 seconds...'):
        try:
            ok = send_alert_email(
                sender.strip(),
                password_clean,
                recipient.strip(),
                df=df,
                scan_time=scan_time or datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                attach_pdf=attach_pdf
            )
        except Exception as e:
            ok = False
            st.session_state['_email_last_error'] = traceback.format_exc()

    if ok:
        st.success(f'✅ Email sent successfully to **{recipient}**!')
        st.balloons()
        st.session_state.pop('_email_last_error', None)
    else:
        st.error('❌ Email failed to send. See the troubleshooting section below.')

        err = st.session_state.get('_email_last_error', '')
        with st.expander('🔍 Full error details'):
            if err:
                st.code(err)
            st.markdown("""
**Common fixes:**

1. **Wrong password type** — You must use a Gmail App Password, NOT your regular Gmail password.
   - Go to [myaccount.google.com → Security → App passwords](https://myaccount.google.com/apppasswords)
   - Create a new App password → select "Mail" → copy the 16-char password

2. **2-Step Verification not enabled** — App passwords only work when 2FA is on.
   - Go to [myaccount.google.com → Security → 2-Step Verification](https://myaccount.google.com/signinoptions/twosv)

3. **Spaces in the app password** — Google shows it as `xxxx xxxx xxxx xxxx`. Remove all spaces before pasting.

4. **Wrong sender address** — The sender must be the same Gmail account you generated the App Password for.

5. **Firewall / network blocking port 587** — Try on a different network or check your firewall rules.
""")

st.divider()

# ── Preview table ─────────────────────────────────────────────────────────────
st.subheader('👀 What will be emailed')
st.caption(f'{len(df)} findings from scan at {scan_time or "unknown time"} — sorted by risk score')

show_cols = [c for c in [
    'ip', 'port', 'service', 'severity', 'risk_score', 'country', 'recommendation'
] if c in df.columns]

st.dataframe(
    df[show_cols].sort_values('risk_score', ascending=False),
    use_container_width=True,
    hide_index=True,
    column_config={
        'risk_score': st.column_config.ProgressColumn(
            'Risk Score', min_value=0, max_value=10, format='%.2f'
        ),
    }
)

# ── Debug expander ────────────────────────────────────────────────────────────
with st.expander('🔧 Debug info'):
    st.write(f'**Sender:** `{sender or "empty"}`')
    st.write(f'**Recipient:** `{recipient or "empty"}`')
    pw_clean_len = len(password.replace(' ', '').strip()) if password else 0
    st.write(f'**Password length (spaces removed):** {pw_clean_len} chars '
             f'{"✅" if pw_clean_len == 16 else "❌ (should be 16)"}')
    st.write(f'**Scan time:** {scan_time or "not set"}')
    st.write(f'**Total findings:** {len(df)}')
    st.write(f'**Critical / High / Medium / Low:** {crit} / {high} / {med} / {low}')
    st.write(f'**Attach PDF:** {attach_pdf}')
    st.write(f'**Loaded from u.env — sender:** `{"set" if ENV_SENDER else "missing"}`')
    st.write(f'**Loaded from u.env — password:** `{"set (" + str(len(ENV_PASS)) + " chars)" if ENV_PASS else "missing"}`')
    st.write(f'**Loaded from u.env — recipient:** `{"set" if ENV_RCPT else "missing"}`')

Overwriting dashboard/pages/9_Email_Alert.py


#Checking Everyfiles  are Working Fine are Not here 

In [21]:
import os, ast

files = [
    ('modules/__init__.py',                'Package marker'),
    ('modules/scanner.py',                 'Nmap + VT extraction'),
    ('modules/analyser.py',                '3-dimensional scoring'),
    ('modules/database.py',                'SQLite history store'),
    ('modules/emailer.py',                 'HTML email + PDF'),
    ('dashboard/app.py',                   'Streamlit main page'),
    ('dashboard/pages/1_Overview.py',      'Overview page'),
    ('dashboard/pages/2_Analysis.py',      'Analysis page'),
    ('dashboard/pages/3_History.py',       'History page'),
    ('dashboard/pages/4_Settings.py',      'Settings page'),
    ('dashboard/pages/5_Scan_Data.py',     'Scan data page'),
    ('dashboard/pages/6_Host_Summary.py',  'Host summary page'),
    ('dashboard/pages/7_Scan_History.py',  'Scan history page'),
    ('dashboard/pages/8_Export.py',        'Export page'),
    ('dashboard/pages/9_Email_Alert.py',   'Email alert page'),
    ('api.py',                             'FastAPI service'),
    ('.streamlit/config.toml',             'Streamlit theme'),
]

print('FILE VERIFICATION')
print('─' * 60)
all_ok = True
for filepath, desc in files:
    if not os.path.exists(filepath):
        print(f'  ❌  {filepath:<45} MISSING')
        all_ok = False
        continue
    size = os.path.getsize(filepath)
    if filepath.endswith('.py') and size > 0:
        try:
            ast.parse(open(filepath, encoding='utf-8').read())
            print(f'  ✅  {filepath:<45} {size:>6}b  {desc}')
        except SyntaxError as e:
            print(f'  ❌  {filepath:<45} SYNTAX ERROR line {e.lineno}: {e.msg}')
            all_ok = False
    else:
        print(f'  ✅  {filepath:<45} {size:>6}b  {desc}')

print('─' * 60)
print('All files OK ✅' if all_ok else '⚠️  Fix errors above')

FILE VERIFICATION
────────────────────────────────────────────────────────────
  ✅  modules/__init__.py                                0b  Package marker
  ✅  modules/scanner.py                              3651b  Nmap + VT extraction
  ✅  modules/analyser.py                             8037b  3-dimensional scoring
  ✅  modules/database.py                             2378b  SQLite history store
  ✅  modules/emailer.py                             12632b  HTML email + PDF
  ✅  dashboard/app.py                                7332b  Streamlit main page
  ✅  dashboard/pages/1_Overview.py                   8000b  Overview page
  ✅  dashboard/pages/2_Analysis.py                   4961b  Analysis page
  ✅  dashboard/pages/3_History.py                    4158b  History page
  ✅  dashboard/pages/4_Settings.py                   3338b  Settings page
  ✅  dashboard/pages/5_Scan_Data.py                  3537b  Scan data page
  ✅  dashboard/pages/6_Host_Summary.py               2197b  Host summary pa

In [20]:
import os
os.makedirs('.streamlit', exist_ok=True)
open('.streamlit/config.toml','w').write(
    '[theme]\n'
    'primaryColor="#4f8ef7"\n'
    'backgroundColor="#0f1117"\n'
    'secondaryBackgroundColor="#1a1d2e"\n'
    'textColor="#e2e8f0"\n'
    'font="sans serif"\n'
)
print('✅ .streamlit/config.toml written')

✅ .streamlit/config.toml written


In [22]:
import os, glob, subprocess, threading, time, requests, sys

# Clear old XML cache
for f in glob.glob('scan_results/*.xml'):
    os.remove(f)
    print(f'🗑️ Deleted: {f}')

# Restart Streamlit
subprocess.run('taskkill /f /im streamlit.exe 2>nul', shell=True)
time.sleep(3)

threading.Thread(
    target=lambda: subprocess.run(
        [sys.executable, '-m', 'streamlit', 'run', 'dashboard/app.py',
         '--server.port', '8501', '--server.headless', 'true'],
    ), daemon=True
).start()

time.sleep(6)

try:
    requests.get('http://localhost:8501/', timeout=5)
    print('✅ Streamlit restarted!')
    print('👉 Open → http://localhost:8501')
except:
    print('⚠️ Wait 10 seconds then open → http://localhost:8501')

🗑️ Deleted: scan_results\nmap.org.xml
🗑️ Deleted: scan_results\scanme.nmap.org.xml
🗑️ Deleted: scan_results\scanme2.nmap.org.xml
✅ Streamlit restarted!
👉 Open → http://localhost:8501


In [25]:
import subprocess, sys, time
subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'dashboard/app.py',
                  '--server.port', '8501', '--server.headless', 'true'])
time.sleep(5)
print('✅ Streamlit restarted!')
print('👉 Open → http://localhost:8501')

✅ Streamlit restarted!
👉 Open → http://localhost:8501
